# Adim 3 — Spark Structured Streaming ile Veri Okuma

Bu notebook, Kafka'dan gelen streaming veriyi Spark Structured Streaming ile okuyup,
temizleyip Delta Lake tablolarina yazar.

**Katmanlar:**
- **Bronze**: Ham veri (Kafka'dan geldigi gibi)
- **Silver**: Temizlenmis veri (null, duplike, format hatalari giderilmis)

---

## 3.1 Sabitler ve Yapilandirma

In [22]:
import os

# Kafka
KAFKA_BOOTSTRAP_SERVERS = "kafka:29092"
KAFKA_TOPIC = "taxi_trips"

# Delta Lake yollari
DATA_DIR = "/home/jovyan/work/data"
DELTA_BASE = os.path.join(DATA_DIR, "delta", "taxi_trips")
CHECKPOINT_BASE = os.path.join(DATA_DIR, "delta", "_checkpoints")

BRONZE_PATH = os.path.join(DELTA_BASE, "bronze")
SILVER_PATH = os.path.join(DELTA_BASE, "silver")
BRONZE_CHECKPOINT = os.path.join(CHECKPOINT_BASE, "bronze")
SILVER_CHECKPOINT = os.path.join(CHECKPOINT_BASE, "silver")

print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")

Bronze: /home/jovyan/work/data/delta/taxi_trips/bronze
Silver: /home/jovyan/work/data/delta/taxi_trips/silver


## 3.2 Spark Session Olusturma

In [23]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC-Taxi-Pipeline") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0,"
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark baslatildi.")
print(f"Spark version: {spark.version}")

Spark baslatildi.
Spark version: 3.5.0


## 3.3 Kafka Mesaj Semasi

Kafka'dan gelen JSON mesajlarinin sema tanimi:

In [24]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType,
    IntegerType, TimestampType
)

taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
])

print("Sema tanimlandi.")
for field in taxi_schema.fields:
    print(f"  {field.name}: {field.dataType.simpleString()}")

Sema tanimlandi.
  VendorID: int
  tpep_pickup_datetime: string
  tpep_dropoff_datetime: string
  passenger_count: double
  trip_distance: double
  RatecodeID: double
  store_and_fwd_flag: string
  PULocationID: int
  DOLocationID: int
  payment_type: int
  fare_amount: double
  extra: double
  mta_tax: double
  tip_amount: double
  tolls_amount: double
  improvement_surcharge: double
  total_amount: double
  congestion_surcharge: double
  Airport_fee: double


## 3.4 Kafka'dan Stream Okuma (Bronze Katman)

Kafka'dan gelen ham veriyi parse edip Bronze katmana yaziyoruz.

In [25]:
from pyspark.sql.functions import from_json, col

# Kafka'dan stream oku
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()

# JSON parse et
bronze_df = raw_stream.select(
    from_json(
        col("value").cast("string"),
        taxi_schema
    ).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

print("Bronze katman hazir.")
bronze_df.printSchema()

Bronze katman hazir.
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



In [26]:
# Bronze'u Delta Lake'e yaz
bronze_query = bronze_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", BRONZE_CHECKPOINT) \
    .outputMode("append") \
    .start(BRONZE_PATH)

print(f"Bronze Delta Lake'e yaziliyor: {BRONZE_PATH}")

Bronze Delta Lake'e yaziliyor: /home/jovyan/work/data/delta/taxi_trips/bronze


## 3.5 Veri Temizleme (Silver Katman)

Bronze verideki sorunlari temizliyoruz:
- `trip_distance <= 0` kayitlar filtrelenir
- `total_amount <= 0` veya `>= 500` kayitlar filtrelenir
- `passenger_count <= 0` kayitlar filtrelenir
- Null datetime kayitlar filtrelenir
- Duplike kayitlar kaldirilir
- Datetime string'leri Timestamp'e donusturulur

In [27]:
from pyspark.sql.functions import hour, dayofweek, when

# Filtreleme ve temizlik
silver_df = bronze_df \
    .filter(
        (col("trip_distance") > 0) &
        (col("total_amount") > 0) &
        (col("total_amount") < 500) &
        (col("passenger_count") > 0) &
        (col("tpep_pickup_datetime").isNotNull()) &
        (col("tpep_dropoff_datetime").isNotNull())
    ) \
    .withColumn(
        "tpep_pickup_datetime",
        col("tpep_pickup_datetime").cast(TimestampType())
    ) \
    .withColumn(
        "tpep_dropoff_datetime",
        col("tpep_dropoff_datetime").cast(TimestampType())
    ) \
    .dropDuplicates(["tpep_pickup_datetime", "PULocationID", "DOLocationID", "total_amount"])

print("Silver katman hazir.")

Silver katman hazir.


In [28]:
# Silver'i Delta Lake'e yaz
silver_query = silver_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", SILVER_CHECKPOINT) \
    .outputMode("append") \
    .start(SILVER_PATH)

print(f"Silver Delta Lake'e yaziliyor: {SILVER_PATH}")

Silver Delta Lake'e yaziliyor: /home/jovyan/work/data/delta/taxi_trips/silver


## 3.6 Stream Durumunu Kontrol Etme

In [29]:
# Aktif stream'lerin durumu
for s in spark.streams.active:
    print(f"Stream: {s.name or s.id}")
    print(f"  Status: {s.status}")
    print(f"  Recent progress: {s.recentProgress[-1] if s.recentProgress else 'N/A'}")
    print()

Stream: 26d3b3ec-149a-48c4-8e8d-78d421fb6b14
  Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': True}
  Recent progress: N/A

Stream: a6f828db-c9f6-4439-9376-773ecf1231ab
  Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
  Recent progress: N/A



## 3.7 Delta Lake Dogrulama

Yazilan verileri batch modda okuyarak dogrulayalim.

In [30]:
# Bronze tablo
bronze_check = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze kayit sayisi: {bronze_check.count():,}")
bronze_check.show(5, truncate=False)

Bronze kayit sayisi: 77,600
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|kafka_timestamp        |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------+
|1       |2024-01-22T13:35:44 |2024-01-22T13:51:14  |2.0            |4.3          |1.0 

In [31]:
# Silver tablo
silver_check = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver kayit sayisi: {silver_check.count():,}")
silver_check.show(5, truncate=False)

Silver kayit sayisi: 41,310
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|kafka_timestamp        |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------+
|2       |2024-01-22 00:40:13 |2024-01-22 00:52:07  |1.0            |2.74         |1.0 

## 3.8 Stream'leri Durdurma

Yeterli veri yazildiktan sonra stream'leri durdurun:

In [32]:
# Tum stream'leri durdur
for s in spark.streams.active:
    print(f"Durduruluyor: {s.id}")
    s.stop()

print("Tum stream'ler durduruldu.")

Durduruluyor: 26d3b3ec-149a-48c4-8e8d-78d421fb6b14
Durduruluyor: a6f828db-c9f6-4439-9376-773ecf1231ab
Tum stream'ler durduruldu.


## 3.9 Bekleme (Opsiyonel)

Stream'lerin tamamlanmasini beklemek icin asagidaki hucreyi calistirin.
**Not:** Bu hucre calisan stream'ler durmayana kadar bloklar.

In [33]:
# spark.streams.awaitAnyTermination()